# Plot model structures with torchview

This notebook visualizes the EMG2QWERTY model architectures using [torchview](https://github.com/mert-kurttutan/torchview). Run from the **project root** (the folder containing `config/` and `emg2qwerty/`).

## 1. Install torchview and graphviz

In [1]:
# Install torchview and graphviz (required for rendering). Restart kernel after install if needed.
!pip install torchview graphviz

  Using cached torchview-0.2.7-py3-none-any.whl.metadata (13 kB)
  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
Using cached torchview-0.2.7-py3-none-any.whl (26 kB)
Using cached graphviz-0.21-py3-none-any.whl (47 kB)



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Setup path and imports

In [2]:
import importlib
import sys
from pathlib import Path

import torch
import hydra
from hydra.utils import instantiate
from omegaconf import OmegaConf

# Project root (folder containing config/ and emg2qwerty/)
ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from torchview import draw_graph

# Optional: use PNG in Jupyter so graphs render well in VS Code
try:
    import graphviz
    graphviz.set_jupyter_format("png")
except Exception:
    pass

## 3. Input shape and helper to instantiate a Lightning module

In [3]:
# Spectrogram input shape: (T, N, bands, channels, freq)
# T=time, N=batch, bands=2 (e.g. left/right), channels=16, freq=33 (n_fft//2+1 for n_fft=64)
T, N = 200, 2
BANDS, CHANNELS, FREQ = 2, 16, 33
INPUT_SHAPE = (T, N, BANDS, CHANNELS, FREQ)


def _hydra_resolver(path: str):
    """Dummy hydra resolver for notebook (HydraConfig is not set when using compose API)."""
    root = str(ROOT)
    fake = {
        "runtime": {"output_dir": root, "cwd": root},
        "run": {"dir": root},
        "job": {"num": 0, "override_dirname": ""},
    }
    v = fake
    for p in path.split("."):
        v = v[p]
    return v


OmegaConf.register_new_resolver("hydra", _hydra_resolver, replace=True)


def get_lightning_module(model_name: str):
    """Load config and instantiate the Lightning module for the given model (e.g. 'tds_conv_ctc')."""
    config_dir = ROOT / "config"
    with hydra.initialize_config_dir(config_dir=str(config_dir), version_base=None):
        cfg = hydra.compose(config_name="base", overrides=[f"model={model_name}"])
        OmegaConf.resolve(cfg)

        target = OmegaConf.select(cfg.module, "_target_", default=None)
        if isinstance(target, str) and "emg2qwerty.lightning" in target:
            mod_path, cls_name = target.rsplit(".", 1)
            mod = importlib.import_module(mod_path)
            LightningClass = getattr(mod, cls_name)
            module = instantiate(
                cfg.module,
                _target_=LightningClass,
                optimizer=cfg.optimizer,
                lr_scheduler=cfg.lr_scheduler,
                decoder=cfg.decoder,
                _recursive_=False,
            )
        else:
            module = instantiate(
                cfg.module,
                optimizer=cfg.optimizer,
                lr_scheduler=cfg.lr_scheduler,
                decoder=cfg.decoder,
                _recursive_=False,
            )
        module = module.eval()
    return module

## 4. Draw graph for a chosen model

We draw the **forward path** of the Lightning module (input → encoder → logits). Use `device='meta'` so no real tensors are allocated. You can change `model_name` and re-run to plot different architectures.

In [5]:
model_name = "tds_conv_ctc"  # options: tds_conv_ctc, tds_conv_crctc, cnn_bilstm_ctc, cnn_bilstm_crctc, cnn_transformer_ctc, conformer_ctc

module = get_lightning_module(model_name)

# Draw full Lightning module forward (one tensor in -> logits out)
graph = draw_graph(
    module,
    input_size=INPUT_SHAPE,
    device="meta",
    graph_name=model_name,
    depth=4,
    expand_nested=True,
    show_shapes=True,
    save_graph=True,
    filename=model_name,
    directory=str(ROOT / "model_graphs"),
)
graph.visual_graph

InterpolationResolutionError: ValueError raised while resolving interpolation: HydraConfig was not set

## 5. Optional: Draw only the core encoder (simpler graph)

For a cleaner picture of the encoder only (no CTC head / loss / metrics), we can draw the underlying `nn.Module` that maps spectrogram → logits. Structure differs by model (TDS has `.model`, CNN+Transformer/Conformer use `.front_end` + `.encoder` + `.head`).

In [6]:
model_name = "tds_conv_ctc"
module = get_lightning_module(model_name)

# TDS and CNN+BiLSTM: single .model (spectrogram -> logits)
if hasattr(module, "model") and module.model is not None:
    core = module.model
    core_name = f"{model_name}_encoder"
# CNN+Transformer / Conformer: front_end -> encoder -> head
else:
    # Wrap in a tiny Module so one forward shows the full pipeline
    class _EncoderWrapper(torch.nn.Module):
        def __init__(self, front_end, encoder, head):
            super().__init__()
            self.front_end = front_end
            self.encoder = encoder
            self.head = head
        def forward(self, x):
            x = self.front_end(x)
            x = self.encoder(x)
            return self.head(x)
    core = _EncoderWrapper(module.front_end, module.encoder, module.head)
    core_name = f"{model_name}_encoder"

graph = draw_graph(
    core,
    input_size=INPUT_SHAPE,
    device="meta",
    graph_name=core_name,
    depth=5,
    expand_nested=True,
    show_shapes=True,
    save_graph=True,
    filename=core_name,
    directory=str(ROOT / "model_graphs"),
)
graph.visual_graph

InterpolationResolutionError: ValueError raised while resolving interpolation: HydraConfig was not set

## 6. Batch-plot all models and save to `model_graphs/`

In [7]:
MODEL_NAMES = [
    "tds_conv_ctc",
    "tds_conv_crctc",
    "cnn_bilstm_ctc",
    "cnn_bilstm_crctc",
    "cnn_transformer_ctc",
    "conformer_ctc",
]

out_dir = ROOT / "model_graphs"
out_dir.mkdir(parents=True, exist_ok=True)

for name in MODEL_NAMES:
    try:
        module = get_lightning_module(name)
        if hasattr(module, "model") and module.model is not None:
            core = module.model
        else:
            class _Enc(torch.nn.Module):
                def __init__(self, fe, enc, h):
                    super().__init__()
                    self.front_end, self.encoder, self.head = fe, enc, h
                def forward(self, x):
                    return self.head(self.encoder(self.front_end(x)))
            core = _Enc(module.front_end, module.encoder, module.head)
        draw_graph(
            core,
            input_size=INPUT_SHAPE,
            device="meta",
            graph_name=name,
            depth=5,
            expand_nested=True,
            show_shapes=True,
            save_graph=True,
            filename=name,
            directory=str(out_dir),
        )
        print(f"Saved: {out_dir / name}.png")
    except Exception as e:
        print(f"{name}: {e}")

print("Done.")

tds_conv_ctc: ValueError raised while resolving interpolation: HydraConfig was not set
tds_conv_crctc: ValueError raised while resolving interpolation: HydraConfig was not set
cnn_bilstm_ctc: ValueError raised while resolving interpolation: HydraConfig was not set
cnn_bilstm_crctc: ValueError raised while resolving interpolation: HydraConfig was not set
cnn_transformer_ctc: ValueError raised while resolving interpolation: HydraConfig was not set
conformer_ctc: ValueError raised while resolving interpolation: HydraConfig was not set
Done.
